# 02.4 — What the framework would have hidden

You've now built a RAG pipeline by hand. Most people don't — they start with
LangChain or LlamaIndex, get something working in fifteen lines, and never see
the parts.

This notebook builds the same thing with LangChain. Not as a warning: you'll
want the framework for real work, and freelance clients ask for it by name. The
point is that going through the primitives first means you can now see what the
framework decided for you.

In [1]:
# Install if using jupyterlab locally on cpu
!pip install -q torch==2.14.0+cpu --index-url https://download.pytorch.org/whl/cpu

In [2]:
!pip install -q langchain==1.4.0 langchain-huggingface==1.2.2 \
               langchain-text-splitters==1.1.2 langchain-openai==1.6.0 \
               pymupdf4llm==1.28.2 sentence-transformers==6.0.1

## The whole thing, in a framework

In [3]:
import os
from langchain_core.documents import Document
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_openai import ChatOpenAI
from pathlib import Path

CORPUS = Path('../../corpus/docs')
NATIVE_PDFS = [
    'sahel-employee-handbook-2023.pdf',
    'sahel-employee-handbook-2025.pdf',
    'sahel-procurement-policy-v3.pdf',
    'nfsc-circular-2024-07-cybersecurity.pdf',
    'nfsc-circular-2025-02-amendment.pdf',
    'kaduna-agro-annual-report-2024.pdf',
    'kaduna-agro-board-minutes-2024-10-17.pdf',
]

# Loaded with the same pinned parser as notebooks 1-3, then wrapped as
# LangChain Documents. See the note below on why we don't use a loader here.
import pymupdf4llm

docs = [
    Document(
        page_content=pymupdf4llm.to_markdown(str(CORPUS / name)),
        metadata={'source': name},
    )
    for name in NATIVE_PDFS
]

splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
splits = splitter.split_documents(docs)

store = InMemoryVectorStore.from_documents(
    splits,
    HuggingFaceEmbeddings(model_name='BAAI/bge-small-en-v1.5'),
)
retriever = store.as_retriever(search_kwargs={'k': 3})

print(f'{len(docs)} pages -> {len(splits)} chunks')

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

7 pages -> 98 chunks


In [4]:
prompt = ChatPromptTemplate.from_template(
    'Answer using only the context below. If it is not there, say you don\'t know.\n\n'
    'CONTEXT:\n{context}\n\nQUESTION: {question}\n\nANSWER:'
)

llm = ChatOpenAI(
    model='minimax/minimax-m2.7:free',
    base_url='https://openrouter.ai/api/v1',
    api_key=os.environ['OPENROUTER_API_KEY'],
    temperature=0,
)

chain = (
    {'context': retriever, 'question': RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print(chain.invoke('What is the maximum emergency procurement without competitive sourcing?'))

Based on the context provided, the maximum emergency procurement without competitive sourcing is **NGN 5,000,000** (five million Nigerian Naira). The Head of Administration may authorize this amount for emergency situations such as failure of critical services that would result in branch closure, loss of connectivity to the core banking application, or breach of a regulatory deadline.


Three notebooks of work, in about twenty-five lines.

That's a genuine saving and you should take it. But every line above made a
decision on your behalf, and the decisions are the subject of this course.

## Decision one: how to split

You wrote `text[i:i+500]`. `RecursiveCharacterTextSplitter` does something
smarter. Look at what.

In [5]:
print('separators tried, in order:', splitter._separators)

separators tried, in order: ['\n\n', '\n', ' ', '']


It tries to split on paragraph breaks first. Failing that, line breaks. Then
spaces. Only as a last resort does it cut mid-word.

That's better than what you wrote. Watch it fix a specific bug from notebook 1.

In [6]:
import pymupdf4llm

raw = pymupdf4llm.to_markdown(str(CORPUS / 'sahel-employee-handbook-2025.pdf'))

naive = [raw[i:i + 500] for i in range(0, len(raw), 500)]
smart = splitter.split_text(raw)

print('fixed-size:')
for c in naive:
    if '25 working days' in c:
        print(' ', repr(c[-70:]))

print('\nrecursive:')
for c in smart:
    if '25 working days' in c:
        print(' ', repr(c))

fixed-size:
  'taff are entitled to **25 working days** of paid annual leave each cal'

recursive:
  '## **4. Annual leave** \n\nConfirmed staff are entitled to **25 working days** of paid annual leave each calendar year, exclusive of public holidays. Leave accrues monthly and may be taken from the seventh month of service. A maximum of 10 working days may be carried into the following year and must be exhausted by 31 March, after which the balance lapses without payment in lieu.'


The fixed-size version cut the sentence in half. The
recursive splitter keeps the heading, the entitlement, the carryover cap and
the deadline together in one chunk.

The framework's default is better than yours. Say so plainly — that's why
people use it.

## But a better default is still a default

You did not choose `chunk_size=500`. You did not choose `chunk_overlap=50`. You
did not choose that separator list. Someone did, for the average case, without
ever seeing your documents.

Here's where that shows.

In [7]:
report = pymupdf4llm.to_markdown(str(CORPUS / 'kaduna-agro-annual-report-2024.pdf'))

for i, c in enumerate(splitter.split_text(report)):
    if '|Kano|' in c:
        print(f'--- chunk {i} ---')
        print(c[:300])
        print('\ncontains the column headers?', '**State**' in c)
        break

--- chunk 9 ---
|Ogun|Ijebu-Ode|43,379|18,564.4|66%|
|---|---|---|---|---|
|Oyo|Ibadan North|31,487|17,136.7|80%|
|Oyo|Ogbomoso|44,403|19,978.0|41%|
|Oyo|Oyo Town|36,427|15,736.5|43%|
|Kaduna|Kaduna South|14,265|8,375.2|79%|
|Kaduna|Zaria|3,771|2,141.3|61%|
|Kaduna|Kafanchan|30,670|16,325.1|53%|
|Kano|Kano Municipa

contains the column headers? False


Still headless. Rows of numbers with nothing saying which column is volume and
which is utilisation.

The splitter was designed for prose. It knows about paragraphs and sentences and
nothing whatsoever about markdown tables, so it splits one down the middle
without hesitating. Your hand-written version had the same bug — the framework
didn't introduce it, and it didn't fix it either.

This is the pattern worth internalising: **frameworks give you good defaults for
the common case and silently fail the same way you would on the uncommon one.**
Module 04 is about making that a decision rather than a default.

## Decision two: the prompt

This is the one that matters most, and it's the one the framework hides best.

In notebook 3 you printed the prompt and read it. In the chain above, the prompt
is assembled somewhere inside `chain.invoke()` and never shown. If the answer
comes back strange, your first question is what the model actually received —
and there's no obvious place to look.

You can get at it, but you have to know how:

In [8]:
question = 'How many days of annual leave do confirmed staff get?'

# This is what the chain actually sends. In the chain, `retriever` sits in the
# context slot, so the prompt receives its raw return value — a list of Document
# objects — and formats whatever that stringifies to.
retrieved = retriever.invoke(question)

print(prompt.format(context=retrieved, question=question))

Human: Answer using only the context below. If it is not there, say you don't know.

CONTEXT:
[Document(id='cb806d10-ef11-4ad3-a18e-d0e18636d3ef', metadata={'source': 'sahel-employee-handbook-2025.pdf'}, page_content='## **4. Annual leave** \n\nConfirmed staff are entitled to **25 working days** of paid annual leave each calendar year, exclusive of public holidays. Leave accrues monthly and may be taken from the seventh month of service. A maximum of 10 working days may be carried into the following year and must be exhausted by 31 March, after which the balance lapses without payment in lieu.'), Document(id='c88d9e56-a16e-48e0-9dd4-070e2743582d', metadata={'source': 'sahel-employee-handbook-2023.pdf'}, page_content='## **4. Annual leave** \n\nConfirmed staff are entitled to **21 working days** of paid annual leave each calendar year, exclusive of public holidays. Leave accrues monthly and may be taken from the seventh month of service. A maximum of 5 working days may be carried into t

That is a Python `repr` pasted into your prompt.

`Document(metadata={'source': ...}, page_content='...')` — repeated for every
chunk, with the class name, the field names and the punctuation all spending
tokens. The source filenames *are* in there, so the model could cite them, but
nothing about this was designed. It's the default `str()` of an internal object,
and it reached the model because nobody looked.

The hand-written version in notebook 3 was deliberate: each chunk labelled with
its document and chunk number, nothing else. Not because it's clever, but
because you decided what the model would see.

The fix in LangChain is a formatting step between the retriever and the prompt.
It's the idiomatic pattern, and it's easy — but you only reach for it once you
know the default is a repr, and the only way to know that is to print it.

In [9]:
def format_docs(docs):
    return '\n\n'.join(
        f"[{d.metadata.get('source', '?')}]\n{d.page_content}" for d in docs
    )


better = (
    {'context': retriever | format_docs, 'question': RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

print(prompt.format(context=format_docs(retrieved), question=question))

Human: Answer using only the context below. If it is not there, say you don't know.

CONTEXT:
[sahel-employee-handbook-2025.pdf]
## **4. Annual leave** 

Confirmed staff are entitled to **25 working days** of paid annual leave each calendar year, exclusive of public holidays. Leave accrues monthly and may be taken from the seventh month of service. A maximum of 10 working days may be carried into the following year and must be exhausted by 31 March, after which the balance lapses without payment in lieu.

[sahel-employee-handbook-2023.pdf]
## **4. Annual leave** 

Confirmed staff are entitled to **21 working days** of paid annual leave each calendar year, exclusive of public holidays. Leave accrues monthly and may be taken from the seventh month of service. A maximum of 5 working days may be carried into the following year and must be exhausted by 31 March, after which the balance lapses without payment in lieu.

[sahel-employee-handbook-2025.pdf]
## **10. Exit** 

Confirmed staff give

In [10]:
print(chain.invoke('How many days of annual leave do confirmed staff get?'))

Based on the context provided, confirmed staff are entitled to **25 working days** of paid annual leave each calendar year (from the 2025 handbook). 

Note: The 2023 handbook indicates 21 working days, suggesting the leave entitlement was increased in the 2025 version.


## Decision three: which version you got

You may have noticed we didn't use a LangChain document loader above. That was
deliberate, and it's the sharpest example in this notebook.

The natural choice is `langchain-pymupdf4llm`, which wraps the same parser
notebooks 1 to 3 use. But it pins an exact version of that parser — and the
version it pins has a bug. On the annual report it returns less than half the
document, dropping the Chairman's statement, the financial review and the top of
the distribution table, and substituting OCR noise from the chart image. The
upstream project fixed this in the next patch release, but the wrapper still
pins the broken one.

Nothing errors. The pipeline runs, the chunks look plausible, and questions about
the first page of that report quietly become unanswerable.

So installing that loader here would silently change the input to notebook 5,
where you calculate the baseline score for the whole course. Instead we load with
the pinned parser and wrap the result in `Document` objects — which is what a lot
of production LangChain code does anyway, once a loader stops fitting.

The lesson isn't that the package is bad. It's that **a convenience wrapper owns
a dependency you thought you controlled**, and it can hold you on a broken
version of it long after the fix ships. Check what your wrappers pin.

## This code will rot

You may have seen a deprecation warning importing LangChain packages — parts of
it are being sunset in favour of standalone integrations. LangChain is on version
1.x, and a great deal of tutorial code written against 0.x no longer runs. Import
paths have moved repeatedly.

Chunking, embeddings, cosine similarity and prompt construction have not moved.
They work the same way they did five years ago and will work the same way in five
more.

That's the real argument for learning the primitives first. Not that frameworks
are bad — use them, and expect clients to ask for them — but the framework is the
part that changes, and the part you'll be debugging.

## What's next

Everything so far has been judged by reading output and forming an impression.
That works for five questions and falls apart at fifty, and it quietly rewards
whichever answer happens to read well.

Notebook 5 replaces your judgement with a number.